# Build Two-Constraint Instruction Triplets

This notebook combines each seed instruction with the corresponding `org_constraint` and `new_constraint` from `datasets/conflict_instruction.jsonl`.

For each available conflict type, it creates:

- `original_instruction`: seed + original constraint
- `new_instruction`: seed + new constraint
- `conflict_instruction`: seed + original constraint + new constraint

No language model is required.

In [1]:
import json
from collections import Counter
from pathlib import Path


def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "datasets" / "seed_instruction.jsonl").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root containing datasets/.")


REPO_ROOT = find_repo_root()
SEED_FILE = REPO_ROOT / "datasets" / "seed_instruction.jsonl"
CONFLICT_FILE = REPO_ROOT / "datasets" / "conflict_instruction.jsonl"
OUTPUT_FILE = REPO_ROOT / "datasets" / "constraint_triplets.jsonl"

print(f"Repository root: {REPO_ROOT}")
print(f"Output file: {OUTPUT_FILE}")

Repository root: /Users/xinmingwang/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/ConInstruct-master
Output file: /Users/xinmingwang/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/ConInstruct-master/datasets/constraint_triplets.jsonl


In [2]:
CONFLICT_TYPES = {
    1: "Conflicts between Content Constraints",
    2: "Conflicts between Keyword Constraints",
    3: "Conflicts between Keyword Constraints and Phrase Constraints",
    4: "Conflicts between Phrase Constraints",
    5: "Conflicts between Length Constraints",
    6: "Conflicts between Format Constraints",
    7: "Conflicts between Style Constraints",
    8: "Conflicts between Phrase Constraints and Content Constraints",
    9: "Conflicts between Phrase Constraints and Style Constraints",
}


def read_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


seed_rows = read_jsonl(SEED_FILE)
conflict_rows = read_jsonl(CONFLICT_FILE)

if len(seed_rows) != len(conflict_rows):
    raise ValueError(
        f"The files must align by row, but found {len(seed_rows)} seed rows "
        f"and {len(conflict_rows)} conflict rows."
    )

print(f"Loaded {len(seed_rows)} aligned samples.")

Loaded 100 aligned samples.


In [3]:
def append_parts(*parts):
    return " ".join(part.strip() for part in parts if part and part.strip())


triplets = []
skipped = []

for sample_id, (seed_row, conflict_row) in enumerate(
    zip(seed_rows, conflict_rows), start=1
):
    seed_instruction = seed_row["instruction"].strip()

    for conflict_type_idx, conflict_type in CONFLICT_TYPES.items():
        conflict_info = conflict_row.get(conflict_type)
        if not conflict_info:
            skipped.append((sample_id, conflict_type_idx, "missing conflict type"))
            continue

        org_constraint = conflict_info.get("org_constraint", "").strip()
        new_constraint = conflict_info.get("new_constraint", "").strip()
        if not org_constraint or not new_constraint:
            skipped.append((sample_id, conflict_type_idx, "missing constraint text"))
            continue

        triplets.append(
            {
                "sample_id": sample_id,
                "task": seed_row.get("task"),
                "conflict_type_idx": conflict_type_idx,
                "conflict_type": conflict_type,
                "seed_instruction": seed_instruction,
                "org_constraint": org_constraint,
                "new_constraint": new_constraint,
                "original_instruction": append_parts(
                    seed_instruction, org_constraint
                ),
                "new_instruction": append_parts(
                    seed_instruction, new_constraint
                ),
                "conflict_instruction": append_parts(
                    seed_instruction, org_constraint, new_constraint
                ),
            }
        )

print(f"Built {len(triplets)} triplets.")
print(f"Skipped {len(skipped)} unavailable sample/type pairs.")

Built 864 triplets.
Skipped 36 unavailable sample/type pairs.


In [4]:
with OUTPUT_FILE.open("w", encoding="utf-8") as file:
    for item in triplets:
        file.write(json.dumps(item, ensure_ascii=False) + "\n")

counts = Counter(item["conflict_type_idx"] for item in triplets)
print(f"Wrote {len(triplets)} records to {OUTPUT_FILE}")
for conflict_type_idx, conflict_type in CONFLICT_TYPES.items():
    print(f"Type {conflict_type_idx}: {counts[conflict_type_idx]:3d} | {conflict_type}")

Wrote 864 records to /Users/xinmingwang/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/ConInstruct-master/datasets/constraint_triplets.jsonl
Type 1: 100 | Conflicts between Content Constraints
Type 2: 100 | Conflicts between Keyword Constraints
Type 3: 100 | Conflicts between Keyword Constraints and Phrase Constraints
Type 4: 100 | Conflicts between Phrase Constraints
Type 5: 100 | Conflicts between Length Constraints
Type 6: 100 | Conflicts between Format Constraints
Type 7: 100 | Conflicts between Style Constraints
Type 8:  94 | Conflicts between Phrase Constraints and Content Constraints
Type 9:  70 | Conflicts between Phrase Constraints and Style Constraints


In [5]:
preview = triplets[0]
for key in (
    "sample_id",
    "conflict_type_idx",
    "original_instruction",
    "new_instruction",
    "conflict_instruction",
):
    print(f"\n{key}:\n{preview[key]}")


sample_id:
1

conflict_type_idx:
1

original_instruction:
Write an email to my boss telling him that I am quitting. The email must include specific reasons for my resignation, such as career growth opportunities or personal circumstances.

new_instruction:
Write an email to my boss telling him that I am quitting. Do not include any reasons for your resignation in the email.

conflict_instruction:
Write an email to my boss telling him that I am quitting. The email must include specific reasons for my resignation, such as career growth opportunities or personal circumstances. Do not include any reasons for your resignation in the email.
